# Phase-2 Step 1: Data Audit & Verification

**Objective:** Rigorous validation of all datasets before proceeding with Phase-2 experiments.

**Datasets:**
- NASA B0005 (train)
- NASA B0006 (test)
- NASA B0007 (test)

**Verification Checklist:**
- [ ] Cycle counts verified
- [ ] Voltage/current ranges checked
- [ ] Temperature stability confirmed
- [ ] SOH computation validated
- [ ] Data quality assessed
- [ ] Random cycles visualized

**Critical:** DO NOT proceed to Step 2 unless ALL checks pass.

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


import sys
import os

# ============================================================================
# CRITICAL SETUP - DO NOT MODIFY
# ============================================================================

# Change to project root directory
print(f"✓ Working directory: {os.getcwd()}")

# Verify we're in the right place
assert os.path.exists('phase2_config.yaml'), "ERROR: Not in project root!"
assert os.path.exists('NASA_DataSets'), "ERROR: NASA_DataSets not found!"

# Add source paths (insert at beginning to take precedence)

# ============================================================================
# IMPORTS
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# Import our modules
from data_audit import DataAudit
from utils import plot_style_setup

plot_style_setup()

# ============================================================================
# LOAD CONFIGURATION
# ============================================================================
config_path = os.path.join(PROJECT_ROOT, 'phase2_config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("✓ Imports complete")
print(f"✓ Configuration loaded from: {config_path}")
print(f"\nRandom seed: {config['random_seed']}")

# ============================================================================
# VERIFY DATA FILES
# ============================================================================
print("\n" + "="*60)
print("VERIFYING DATA FILES")
print("="*60)

all_files_exist = True
for name, dataset in config['datasets'].items():
    if 'path' in dataset:
        rel_path = dataset['path']
        # Check with absolute path
        abs_path = os.path.join(PROJECT_ROOT, rel_path)
        exists = os.path.exists(abs_path)
        status = "✅" if exists else "❌"
        print(f"  {status} {name}: {rel_path}")
        if exists:
            # Get file size
            if os.path.isfile(abs_path):
                size_mb = os.path.getsize(abs_path) / (1024*1024)
                print(f"       Size: {size_mb:.1f} MB")
        else:
            all_files_exist = False
            print(f"       MISSING: {abs_path}")

print("="*60)
if all_files_exist:
    print("✅ ALL DATA FILES VERIFIED - READY TO PROCEED")
else:
    print("❌ SOME FILES MISSING - CHECK PATHS ABOVE")
print("="*60)

## 1. Initialize Data Auditor

In [ ]:
# Create auditor
auditor = DataAudit(config)

print("Data Auditor initialized")
print(f"Audit directory: {auditor.audit_dir}")
print(f"Timestamp: {auditor.timestamp}")

## 2. Audit NASA B0005 (Training Battery)

In [ ]:
# Audit B0005
b0005_config = config['datasets']['nasa_train']
b0005_result = auditor.audit_nasa_battery(
    b0005_config['path'],
    b0005_config['battery_id']
)

## 3. Audit NASA B0006 (Test Battery 1)

In [ ]:
# Audit B0006
b0006_config = config['datasets']['nasa_test_1']
b0006_result = auditor.audit_nasa_battery(
    b0006_config['path'],
    b0006_config['battery_id']
)

## 4. Audit NASA B0007 (Test Battery 2)

In [ ]:
# Audit B0007
b0007_config = config['datasets']['nasa_test_2']
b0007_result = auditor.audit_nasa_battery(
    b0007_config['path'],
    b0007_config['battery_id']
)

## 5. Consolidated Audit Results

In [ ]:
# Collect all results
all_results = {
    'B0005': b0005_result,
    'B0006': b0006_result,
    'B0007': b0007_result
}

# Create summary table
summary_data = []
for battery_id, result in all_results.items():
    summary_data.append({
        'Battery': battery_id,
        'Status': '✅ PASS' if result['passed'] else '❌ FAIL',
        'Cycles': result['total_cycles'],
        'Warnings': len(result['warnings']),
        'Errors': len(result['errors'])
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*60)
print("AUDIT SUMMARY".center(60))
print("="*60)
print(summary_df.to_string(index=False))
print("="*60)

## 6. Detailed Check Comparison

In [ ]:
# Compare specific checks across batteries
check_names = ['voltage', 'current', 'temperature', 'soh', 'data_points', 'spikes']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, check_name in enumerate(check_names):
    ax = axes[i]
    
    # Extract data for this check
    batteries = []
    statuses = []
    colors = []
    
    for battery_id, result in all_results.items():
        batteries.append(battery_id)
        check = result['checks'].get(check_name, {})
        passed = check.get('passed', False)
        statuses.append(1 if passed else 0)
        colors.append('green' if passed else 'red')
    
    # Bar plot
    ax.bar(batteries, statuses, color=colors, alpha=0.7, edgecolor='black')
    ax.set_ylim([0, 1.2])
    ax.set_ylabel('Status')
    ax.set_title(f'Check: {check_name.upper()}', fontweight='bold')
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['FAIL', 'PASS'])
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Data Audit: Check Status Across Batteries', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✓ All checks visualized")

## 7. SOH Comparison Across Batteries

In [ ]:
# Plot SOH degradation for all batteries
fig, ax = plt.subplots(figsize=(12, 6))

for battery_id, result in all_results.items():
    soh_check = result['checks']['soh']
    
    initial_soh = soh_check['initial']
    final_soh = soh_check['final']
    n_cycles = result['total_cycles']
    
    # Simple linear approximation for visualization
    soh_approx = np.linspace(initial_soh, final_soh, n_cycles)
    
    ax.plot(range(n_cycles), soh_approx, 'o-', label=battery_id, alpha=0.7, markersize=3)

ax.axhline(y=80, color='r', linestyle='--', label='EOL Threshold (80%)', alpha=0.5)
ax.set_xlabel('Cycle Number', fontsize=12)
ax.set_ylabel('SOH (%)', fontsize=12)
ax.set_title('SOH Degradation Comparison (Approximate)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSOH Degradation Summary:")
for battery_id, result in all_results.items():
    soh_check = result['checks']['soh']
    print(f"  {battery_id}: {soh_check['initial']:.2f}% → {soh_check['final']:.2f}% (Δ={soh_check['initial']-soh_check['final']:.2f}%)")

## 8. Final Verdict

In [ ]:
# Check if all batteries passed
all_passed = all(result['passed'] for result in all_results.values())

print("\n" + "#"*80)
print("PHASE-2 STEP 1: FINAL VERDICT".center(80))
print("#"*80)

if all_passed:
    print("\n✅ ALL BATTERIES PASSED AUDIT")
    print("✅ DATA QUALITY VERIFIED")
    print("✅ SOH COMPUTATION VALIDATED")
    print("✅ PROCEED TO STEP 2: MULTI-DATASET HARMONIZATION")
else:
    print("\n❌ AUDIT FAILURES DETECTED")
    print("❌ REVIEW WARNINGS AND ERRORS ABOVE")
    print("❌ FIX ISSUES BEFORE PROCEEDING")
    
    # List failures
    print("\nFailed batteries:")
    for battery_id, result in all_results.items():
        if not result['passed']:
            print(f"  - {battery_id}")
            for error in result['errors']:
                print(f"    Error: {error}")

print("\n" + "#"*80)

# Save final summary
auditor._save_audit_report(all_results, "NASA_batteries")
print("\n✓ Audit reports saved")

## Summary

**Step 1 Checklist:**

✓ B0005 audited  
✓ B0006 audited  
✓ B0007 audited  
✓ Voltage ranges verified  
✓ Current ranges verified  
✓ Temperature stability checked  
✓ SOH computation validated  
✓ Data quality assessed  
✓ Random cycles visualized  
✓ Audit reports generated  

**Next Steps:**
- If all checks passed → Proceed to Step 2 (Harmonization)
- If any failures → Fix issues and re-run audit

**Critical:** Do not proceed unless ALL batteries pass audit!